# Relacionamentos, N+1 e Entregando Dados para o Pandas

---

## Contexto: o pipeline está lento!

O dashboard de vendas está demorando 8 segundos para carregar, sendo o dobro do aceitável.  
Você investiga e descobre: um loop está disparando centenas de queries desnecessárias.  
Esse é o famoso **problema N+1** e você vai resolvê-lo agora.

Nesta aula você vai:
- Entender lazy vs eager loading na prática
- Identificar e corrigir o problema N+1
- Usar `joinedload` e `selectinload`
- Entregar resultados em DataFrames do Pandas

---

## 1. Configuração

In [1]:
from pathlib import Path
from datetime import datetime
from decimal import Decimal
from sqlalchemy import (
    create_engine, select, and_, or_, func, inspect,
    String, DateTime, Numeric, Integer, Boolean, ForeignKey, CheckConstraint, Index
)
from sqlalchemy.orm import (
    DeclarativeBase, Mapped, mapped_column, relationship, Session,
    joinedload, selectinload
)
from models import Base, Cliente, Produto, ItemPedido, Pedido

# Cria o diretório do banco se não existir
Path("../database").mkdir(exist_ok=True)

# Cria a engine para conectar ao banco SQLite
engine = create_engine("sqlite:///../database/datavendas.db")
# Confirmação visual de que a conexão/engine foi configurada com sucesso
print("Conexão OK ✅")

Conexão OK ✅


## 2. Lazy Loading o comportamento padrão

Por padrão, o SQLAlchemy usa **lazy loading**: os dados relacionados só são buscados quando você os acessa.  

Isso é conveniente para acessos pontuais, mas **desastroso em loops**.

In [3]:
# Para ver as queries sendo disparadas, ative o echo temporariamente
engine_verbose = create_engine("sqlite:///../database/datavendas.db", echo=True)

print("=" * 60)
print("LAZY LOADING: observe quantas queries são disparadas:")
print("=" * 60)

with Session(engine_verbose) as session:
    pedidos = session.scalars(select(Pedido)).all()
    # Até aqui: 1 query

    for p in pedidos[:3]:  # Limitando a 3 para não poluir o output
        # AQUI: uma nova query para cada pedido!
        nome_cliente = p.cliente.nome
        print(f"  Pedido #{p.id} → {nome_cliente}")

LAZY LOADING: observe quantas queries são disparadas:
2026-03-24 11:26:36,695 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026-03-24 11:26:36,762 INFO sqlalchemy.engine.Engine SELECT tb_pedidos.id, tb_pedidos.cliente_id, tb_pedidos.data_pedido, tb_pedidos.status, tb_pedidos.valor_total 
FROM tb_pedidos
2026-03-24 11:26:36,766 INFO sqlalchemy.engine.Engine [generated in 0.00398s] ()
2026-03-24 11:26:36,803 INFO sqlalchemy.engine.Engine SELECT tb_clientes.id AS tb_clientes_id, tb_clientes.nome AS tb_clientes_nome, tb_clientes.email AS tb_clientes_email, tb_clientes.cidade AS tb_clientes_cidade, tb_clientes.estado AS tb_clientes_estado, tb_clientes.data_cadastro AS tb_clientes_data_cadastro 
FROM tb_clientes 
WHERE tb_clientes.id = ?
2026-03-24 11:26:36,806 INFO sqlalchemy.engine.Engine [generated in 0.00292s] (1,)
  Pedido #1 → Deborah Foroni
2026-03-24 11:26:36,818 INFO sqlalchemy.engine.Engine SELECT tb_clientes.id AS tb_clientes_id, tb_clientes.nome AS tb_clientes_nome, tb_clientes.email AS tb_clientes_email, tb_clientes.cidade AS tb_clientes_cidade, tb_clientes.estado AS tb_clientes_estado, tb_clientes.

### O problema N+1 em números

Se você tem 200 pedidos e acessa `p.cliente` em loop:

```
1 query para buscar os 200 pedidos
+ 200 queries para buscar o cliente de cada pedido
= 201 queries  ← isso é o N+1!
```

Com banco local, cada query leva ~1ms. Com PostgreSQL em produção, pode levar 10-50ms.  
**200 × 50ms = 10 segundos**  é exatamente isso que estava travando o dashboard.

## 3. Eager Loading — solução para N+1

A solução é avisar ao SQLAlchemy **antes de executar** que você vai precisar dos relacionamentos.  
Ele traz tudo de uma vez, em 1 ou 2 queries.

### 3.1 `joinedload` — usa um JOIN

In [4]:
print("=" * 60)
print("EAGER LOADING com joinedload — observe: 1 única query:")
print("=" * 60)

with Session(engine_verbose) as session:
    stmt = (
        select(Pedido)
        .options(joinedload(Pedido.cliente))  # avisa: traga o cliente junto!
    )
    pedidos = session.scalars(stmt).all()

    for p in pedidos[:3]:
        # Sem queries adicionais — os dados já estão carregados!
        print(f"  Pedido #{p.id} → {p.cliente.nome}")

EAGER LOADING com joinedload — observe: 1 única query:
2026-03-24 11:27:45,870 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-24 11:27:45,882 INFO sqlalchemy.engine.Engine SELECT tb_pedidos.id, tb_pedidos.cliente_id, tb_pedidos.data_pedido, tb_pedidos.status, tb_pedidos.valor_total, tb_clientes_1.id AS id_1, tb_clientes_1.nome, tb_clientes_1.email, tb_clientes_1.cidade, tb_clientes_1.estado, tb_clientes_1.data_cadastro 
FROM tb_pedidos LEFT OUTER JOIN tb_clientes AS tb_clientes_1 ON tb_clientes_1.id = tb_pedidos.cliente_id
2026-03-24 11:27:45,884 INFO sqlalchemy.engine.Engine [generated in 0.00210s] ()
  Pedido #1 → Deborah Foroni
  Pedido #2 → Maria Sophia Guerra
  Pedido #3 → Laís Santos
2026-03-24 11:27:45,897 INFO sqlalchemy.engine.Engine ROLLBACK


### 3.2 `selectinload` — usa WHERE IN

Melhor para **coleções** (um-para-muitos): 1 query principal + 1 query `WHERE id IN (...)`

In [5]:
print("=" * 60)
print("selectinload — 2 queries para N pedidos com itens:")
print("=" * 60)

with Session(engine_verbose) as session:
    stmt = (
        select(Pedido)
        .options(
            joinedload(Pedido.cliente),        # cliente: join
            selectinload(Pedido.itens)         # itens: selectin (coleção)
            .joinedload(ItemPedido.produto)    # produto de cada item: join
        )
    )
    pedidos = session.scalars(stmt).unique().all()

    for p in pedidos[:2]:
        print(f"\nPedido #{p.id} | {p.cliente.nome} | R${p.valor_total}")
        for item in p.itens:
            print(f"  - {item.quantidade}x {item.produto.nome} @ R${item.preco_venda}")

selectinload — 2 queries para N pedidos com itens:
2026-03-24 11:28:52,051 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-24 11:28:52,060 INFO sqlalchemy.engine.Engine SELECT tb_pedidos.id, tb_pedidos.cliente_id, tb_pedidos.data_pedido, tb_pedidos.status, tb_pedidos.valor_total, tb_clientes_1.id AS id_1, tb_clientes_1.nome, tb_clientes_1.email, tb_clientes_1.cidade, tb_clientes_1.estado, tb_clientes_1.data_cadastro 
FROM tb_pedidos LEFT OUTER JOIN tb_clientes AS tb_clientes_1 ON tb_clientes_1.id = tb_pedidos.cliente_id
2026-03-24 11:28:52,066 INFO sqlalchemy.engine.Engine [generated in 0.00616s] ()
2026-03-24 11:28:52,103 INFO sqlalchemy.engine.Engine SELECT tb_itens_pedidos.pedido_id AS tb_itens_pedidos_pedido_id, tb_itens_pedidos.id AS tb_itens_pedidos_id, tb_itens_pedidos.produto_id AS tb_itens_pedidos_produto_id, tb_itens_pedidos.quantidade AS tb_itens_pedidos_quantidade, tb_itens_pedidos.preco_venda AS tb_itens_pedidos_preco_venda, tb_produtos_1.id AS tb_produtos_1_id, tb_


Pedido #1 | Deborah Foroni | R$3499.90
  - 1x Notebook Pro 15 @ R$3499.90

Pedido #2 | Maria Sophia Guerra | R$13120.80
  - 10x Tempora excepturi labore @ R$1312.08
2026-03-24 11:28:52,162 INFO sqlalchemy.engine.Engine ROLLBACK


### Qual estratégia usar?

| Estratégia | Como funciona | Quando usar |
|---|---|---|
| `joinedload()` | 1 query com JOIN | Relacionamentos muitos-para-um (pedido → cliente) |
| `selectinload()` | 1 query + 1 `WHERE IN` | Coleções (cliente → lista de pedidos) |

---

## 4. Entregando resultados para o Pandas

No dia a dia, a maioria das análises termina em um DataFrame.  
O fluxo ideal é: **fazer o recorte no banco** (eficiente) e trazer para Python apenas o necessário.

**Padrão recomendado:**
1. `select()` com as colunas necessárias
2. `mappings()` para obter dicionários
3. `pd.DataFrame()` para montar o DataFrame

In [8]:
# Relatório de vendas por cliente
import pandas as pd  # Importa a biblioteca Pandas para trabalhar com DataFrames
with Session(engine) as session:  # Abre uma sessão para executar a query no banco
    stmt = (  # Monta a query SQLAlchemy para selecionar dados de vendas
        select(  # Seleciona colunas específicas de Cliente e Pedido
            Cliente.nome.label("cliente"),  # Nome do cliente, renomeado para "cliente"
            Cliente.estado,  # Estado do cliente
            Pedido.id.label("pedido_id"),  # ID do pedido, renomeado para "pedido_id"
            Pedido.data_pedido,  # Data do pedido
            Pedido.valor_total,  # Valor total do pedido
            Pedido.status,  # Status do pedido
        )
        .join(Pedido.cliente)  # Faz JOIN entre Pedido e Cliente via relacionamento ORM
        .order_by(Pedido.data_pedido.desc())  # Ordena os resultados por data do pedido decrescente (mais recentes primeiro)
    )

    dados = session.execute(stmt).mappings().all()  # Executa a query e retorna os resultados como uma lista de dicionários

df = pd.DataFrame(dados)  # Converte a lista de dicionários em um DataFrame do Pandas

print("DataFrame de vendas:")  # Imprime cabeçalho
print(df.head())  # Exibe as primeiras 5 linhas do DataFrame
print(f"\nShape: {df.shape}")  # Imprime o formato do DataFrame (linhas, colunas)
print(f"Colunas: {list(df.columns)}")  # Imprime a lista de colunas do DataFrame

DataFrame de vendas:
               cliente estado  pedido_id                data_pedido  \
0       Deborah Foroni     SP          1 2026-03-24 09:52:39.761643   
1  Maria Sophia Guerra     RJ         46 2026-03-24 02:54:10.969851   
2        Stella Guerra     DF         79 2026-03-20 19:56:57.714146   
3          Laís Santos     SP          3 2026-03-14 08:14:14.320558   
4  Mathias Casa Grande     BA         22 2026-03-07 02:24:19.584393   

  valor_total     status  
0     3499.90     Criado  
1   137396.68       Pago  
2    67832.93       Pago  
3   110713.96    Enviado  
4    38436.76  Cancelado  

Shape: (101, 6)
Colunas: ['cliente', 'estado', 'pedido_id', 'data_pedido', 'valor_total', 'status']


In [9]:
# Análise rápida após ter o DataFrame
if not df.empty:
    print("Vendas por status:")
    print(df.groupby("status")["valor_total"].agg(["count", "sum"]).round(2))
    
    print("\nTop 5 clientes por valor total:")
    top_clientes = (
        df.groupby("cliente")["valor_total"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
    )
    print(top_clientes)

Vendas por status:
           count         sum
status                      
Cancelado     26   868558.61
Criado        25   876638.60
Enviado       26  1245300.16
Pago          24  1303040.02

Top 5 clientes por valor total:
cliente
Maria Sophia Guerra    422479.56
Théo Cirino            412942.75
Stella Guerra          359972.12
Luiz Miguel Fogaça     333001.78
Mathias Casa Grande    329982.24
Name: valor_total, dtype: object


---

## Exercício: Usando IA para isso

**Prompt para gerar pipeline Pandas:**
```
Crie um pipeline SQLAlchemy + Pandas que:
1. Busque os pedidos do último mês com cliente e itens (sem N+1)
2. Converta para DataFrame
3. Calcule receita total por categoria de produto
4. Retorne o top 5 categorias

Modelos ORM: [cole os modelos]
```

### Resposta:Código gerado pelo Copilot


In [11]:
from sqlalchemy import select, func
from datetime import datetime, timedelta
import pandas as pd

# Calcula o último mês
hoje = datetime.now()
ultimo_mes = hoje.replace(day=1) - timedelta(days=1)
inicio_ultimo_mes = ultimo_mes.replace(day=1)
fim_ultimo_mes = ultimo_mes

with Session(engine) as session:
    # Query para buscar pedidos do último mês com cliente e itens (JOINs explícitos evitam N+1)
    stmt = (
        select(
            Pedido.id.label("pedido_id"),
            Pedido.data_pedido,
            Pedido.valor_total,
            Cliente.nome.label("cliente_nome"),
            Cliente.estado,
            ItemPedido.quantidade,
            ItemPedido.preco_venda,
            Produto.nome.label("produto_nome"),
            Produto.categoria,
        )
        .join(Pedido.cliente)  # JOIN com Cliente
        .join(Pedido.itens)    # JOIN com ItemPedido
        .join(ItemPedido.produto)  # JOIN com Produto
        .where(Pedido.data_pedido >= inicio_ultimo_mes)  # Filtra pedidos do último mês
        .where(Pedido.data_pedido <= fim_ultimo_mes)
        # Removidas .options() pois a query é expression-based
    )
    
    dados = session.execute(stmt).mappings().all()

# Converte para DataFrame
df = pd.DataFrame(dados)

# Calcula receita total por categoria (soma de quantidade * preco_venda por categoria)
df['receita_item'] = df['quantidade'] * df['preco_venda']
receita_por_categoria = df.groupby('categoria')['receita_item'].sum().reset_index()

# Retorna top 5 categorias por receita total
top_5_categorias = receita_por_categoria.sort_values('receita_item', ascending=False).head(5)

print("Top 5 categorias por receita total no último mês:")
top_5_categorias

Top 5 categorias por receita total no último mês:


,categoria,receita_item
2,Livros,153423.98
0,Eletrônicos,66106.87
4,Roupas,43642.52
3,Móveis,25032.15
1,Esportes,24391.50
